# Pour_Zone_CNT PV 기반 인코더 rate 이상치 탐지

`Pour_Zone_CNT PV` 를 "속도"로 쓰기 위해, 다음 전처리를 먼저 적용한다:

1. **리셋(음수 diff) 제외** — 140에서 6으로 떨어지는 건 진짜 감소가 아니라 순환 카운터의 한 바퀴다.
   순환길이를 정확히 모르는 채로 값을 만들어내면(unwrap) 그 추정이 틀렸을 때 가짜 이상치가 생긴다
   (실제로 확인됨). 그래서 **이 한 칸의 diff는 그냥 계산하지 않는다** — 리셋 직전 값은 버리고,
   리셋 직후 값부터 새로 정상적으로 diff를 이어간다.
2. **source(PREDICTIVE/DIAGNOSTIC) 전환 경계 제외** — 서로 다른 트랙(예측치 vs 진단치)을 이어붙이는 diff 는
   실제 속도가 아니라 서로 다른 값을 읽은 것이라 이 한 칸만 버린다(값은 그대로 유지, diff만 제외).

그 다음 `rate = diff / dt` (pulse/초) 를 계산해서 **평균과 표준편차**로 이상치를 찾는다. 몇 표준편차 위부터
이상치로 볼지는 미리 정하지 않고, 임계값을 바꿔가며 개수가 어떻게 변하는지 표로 보고 고른다.

리셋과 source 경계를 전부 걷어냈기 때문에, 여기서 나오는 이상치는 전부 "만들어낸 값" 없이 **실제로 관측된
diff/dt** 뿐이다 — unwrap 오차 같은 걸 따로 걱정할 필요가 없다.

## 1. 모듈 로드 및 파라미터

In [ ]:
import os
import sys
from datetime import datetime

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
%matplotlib inline
matplotlib.rcParams["font.family"] = "Malgun Gothic"
matplotlib.rcParams["axes.unicode_minus"] = False

ROOT_DIR = os.path.dirname(os.path.dirname(os.getcwd()))
sys.path.insert(0, ROOT_DIR)

from Log_Extractor import LogExtractor

ENV_PATH = os.path.join(ROOT_DIR, ".env")
SAVE_DIR = os.path.join(os.getcwd(), "extracted_csv")

pd.set_option("display.width", 200)
print(f"ROOT_DIR = {ROOT_DIR}")
print(f"ENV_PATH = {ENV_PATH}  (존재={os.path.exists(ENV_PATH)})")

In [ ]:
TAGS = [
    "Conv_Emergency",
    "Pour_Zone_CNT PV",
    "Conv_Brake_Time",
]

START_TIME = "-3d"
END_TIME = "now()"

N_STD = 5   # 이상치 기준: 평균 + N_STD * 표준편차. 아래 스윗 표 보고 조정하면 된다.

## 2. 추출

In [ ]:
extractor = LogExtractor(env_path=ENV_PATH)
raw = extractor.get_data(start_time=START_TIME, end_time=END_TIME, target_tags=TAGS)
print(f"{raw.shape[0]:,}행 × {raw.shape[1]}컴럼   기간: {raw.index.min()} ~ {raw.index.max()}")

## 3. 전처리 — 리셋 & source 전환 경계 제외 (unwrap 없음)

In [ ]:
raw_diff = raw["Pour_Zone_CNT PV"].diff()
dt = raw.index.to_series().diff().dt.total_seconds()

reset = raw_diff < 0
print(f"리셋 {int(reset.sum()):,}건 — 이 diff는 계산하지 않는다(값을 만들어내지 않음)")

source_change = raw["source"] != raw["source"].shift(1)
source_change.iloc[0] = False
print(f"source 전환 경계: {int(source_change.sum()):,}건 — 이 diff도 속도 계산에서 제외한다")

clean_diff = raw_diff.where((raw_diff >= 0) & (~source_change))
rate = clean_diff / dt
print(f"\n원래 diff 전체: {raw_diff.notna().sum():,}건 → 정리 후 남은 diff: {clean_diff.notna().sum():,}건")

## 4. 평균 · 표준편차

In [ ]:
running = raw["Conv_Brake_Time"] != 0
baseline = rate[running & clean_diff.notna()]

mean_r = baseline.mean()
std_r = baseline.std()
print(f"표본수 {len(baseline):,}건")
print(f"평균 rate = {mean_r:.3f} pulse/초")
print(f"표준편차   = {std_r:.3f} pulse/초")

## 5. 몇 표준편차부터 이상치로 볼지 — 임계값 스윗

In [ ]:
sweep_rows = []
for n in [2, 3, 4, 5, 6, 7, 8, 10, 15, 20, 25, 30]:
    thr = mean_r + n * std_r
    cnt = int((baseline > thr).sum())
    sweep_rows.append({"N(표준편차)": n, "임계값(pulse/초)": round(thr, 2), "이상치 개수": cnt})

sweep = pd.DataFrame(sweep_rows)
sweep

## 6. 시각화 — 시간대별 rate, 평균 점선, 이상치

In [ ]:
thr = mean_r + N_STD * std_r
outlier_mask = running & clean_diff.notna() & (rate > thr)
outlier_idx = rate.index[outlier_mask]
print(f"N_STD={N_STD} → 임계값 {thr:.2f} pulse/초, 이상치 {len(outlier_idx):,}건")

fig, ax = plt.subplots(figsize=(13, 5))
ax.scatter(baseline.index, baseline, s=5, alpha=0.15, color="lightgray", label="정상 범위")
ax.scatter(outlier_idx, rate.loc[outlier_idx], s=45, color="crimson", marker="x", label=f"이상치 (>{N_STD}표준편차)", zorder=5)
ax.axhline(mean_r, color="black", linestyle="--", linewidth=1.2, label=f"평균 ({mean_r:.2f} pulse/초)")
ax.axhline(thr, color="crimson", linestyle=":", linewidth=1, label=f"임계값 (평균+{N_STD}표준편차)")
ax.set_ylabel("rate (pulse/초)")
ax.set_xlabel("시각")
ax.set_title("Pour_Zone_CNT PV rate 이상치 (평균/표준편차 기준)")
ax.legend(fontsize=8)
ax.grid(alpha=0.3)
fig.autofmt_xdate()
fig.tight_layout()

## 7. 보조 시각화 — dt vs diff (왜 이상치인지: dt 가 짧아서인지 diff 자체가 큰건지)

In [ ]:
mask_plot = running & clean_diff.notna()

fig, ax = plt.subplots(figsize=(8, 6))
ax.scatter(dt[mask_plot], clean_diff[mask_plot], s=12, alpha=0.3, color="lightgray", label="정상 범위")
ax.scatter(dt.loc[outlier_idx], clean_diff.loc[outlier_idx], s=45, color="crimson", marker="x", label="이상치", zorder=5)

xlim = dt[mask_plot].quantile(0.995)  # 브레이크로 멈춰서 dt 가 수십~수백초로 튀는 소수를 빼고 정상 구간만 확대해서 본다
xs = np.linspace(0, xlim, 50)
ax.plot(xs, mean_r * xs, color="black", linestyle="--", linewidth=1, label=f"평균 rate 기준선 ({mean_r:.1f} pulse/초)")

ax.set_xlim(0, xlim)
ax.set_xlabel("dt (실제 경과시간, 초)")
ax.set_ylabel("diff (Pour_Zone_CNT PV 증가량)")
ax.set_title("dt vs diff — 기준선보다 위에 있어야 진짜 '빠른' 것")
ax.legend(fontsize=8)
ax.grid(alpha=0.3)
fig.tight_layout()

## 8. 이상치 표

리셋과 source 전환 경계를 이미 걷어냈으므로, 여기 나오는 값은 전부 실제로 관측된 diff/dt다.

In [ ]:
outlier_table = pd.DataFrame({
    "시각": outlier_idx,
    "dt(초)": dt.loc[outlier_idx].to_numpy(),
    "diff": clean_diff.loc[outlier_idx].to_numpy(),
    "rate(pulse/초)": rate.loc[outlier_idx].to_numpy(),
    "source": raw.loc[outlier_idx, "source"].to_numpy(),
}).sort_values("rate(pulse/초)", ascending=False).reset_index(drop=True)

outlier_table